# 🎙️ Channel 99 AI Studio — VoxCPM (Colab)
**Run all (Ctrl+F9)** → share URL ကို copy → recap tool ထဲ ကပ်.
Runtime: **T4 GPU** (Runtime → Change runtime type)


In [ ]:
# — ကိုယ့်အသံ ဒေါင်း (Drive link, login မလို) —
import urllib.request, os
VOICE_PATH = "/tmp/my_voice.wav"
if not os.path.exists(VOICE_PATH):
    print("⬇️ my_voice.wav ဒေါင်းနေပြီ...")
    urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=1zTw0hH17OnZoN8I28SMNnkS02eyTTH8n&confirm=t", VOICE_PATH)
    print(f"✅ {os.path.getsize(VOICE_PATH)/1024:.0f} KB")
else:
    print("✅ my_voice.wav ready")

# — Alex preset အသံ ဒေါင်း (Drive link, login မလို) —
ALEX_PATH = "/tmp/alex_voice.wav"
if not os.path.exists(ALEX_PATH):
    print("⬇️ alex_voice.wav ဒေါင်းနေပြီ...")
    try:
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=1r0dg-Iv9hCVYEwl0PkqoZcJWckB4Lewi&confirm=t", ALEX_PATH)
        print(f"✅ {os.path.getsize(ALEX_PATH)/1024:.0f} KB")
    except Exception as e:
        print(f"❌ alex_voice.wav ဒေါင်းမရပါ ({e}) — Drive sharing (Anyone with the link) စစ်ပါ")
else:
    print("✅ alex_voice.wav ready")


In [ ]:
# Install (ပထမအကြိမ်ပဲ ကြာ, နောက် run တွေ cache)
!pip install -q voxcpm soundfile gradio gradio_client


In [ ]:
import os
# — GPU စစ် (T4 မရရင် ရှင်းရှင်းပြ) —
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError(
        "❌ GPU မတွေ့!\n"
        "1) Runtime → Change runtime type → T4 GPU ရွေးပြီး ပြန် run\n"
        "2) 'GPU quota exhausted' ဆို: account ပြောင်း (သို့) Kaggle သုံး"
    )

torch._dynamo.config.disable = True
from voxcpm import VoxCPM
import gradio as gr
import soundfile as sf
import numpy as np, gc
from datetime import datetime

# Model: Drive မှာ ရှိရင် Drive က, မရှိရင် HuggingFace ကနေ တိုက်ရိုက်ဒေါင်း
_drive_model = "/content/drive/MyDrive/Channel99_Studio/VoxCPM/models"
if os.path.isdir(_drive_model):
    _model_src = _drive_model
    print(f"📁 Drive model သုံးမယ်: {_drive_model}")
else:
    _model_src = "openbmb/VoxCPM2"
    print(f"☁️ Drive မှာ model မတွေ့ — HuggingFace ကနေ ဒေါင်းမယ်: {_model_src}")

print("⏳ Model ခေါ်နေပါပြီ...")
model = VoxCPM.from_pretrained(_model_src, load_denoiser=False)
SR = model.tts_model.sample_rate
print(f"✅ Model ready! sample_rate={SR}")

# Output folder သေချာ ရှိနေအောင်
OUTPUT_DIR = "/tmp/voxcpm_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"📁 outputs ready: {OUTPUT_DIR}")





In [ ]:
# ── ကိုယ့်အသံ preset ──
# my_voice.wav ကို Drive: /Channel99_Studio/ ထဲ တင်
# prompt_text = ဖိုင်ထဲမှာ ပြောထားတဲ့ စာသား အတိအကျ
VOICE_DATABASE = {
    "🎙️ ကိုယ့်အသံ": {
        "audio_path": "/tmp/my_voice.wav",
        "prompt_text": "ရန်ကုန်စည်ပင်နယ်နိမိတ်အတွင်း ဓာတ်လှေကားပါဝင်မယ့် ဆောက်လုပ်မယ့် အဆောက်အဦးတွေနဲ့ ဆောက်လုပ်ဆဲ အဆောက်အဦးတွေမှာ ဆိုလာဓာတ်အားပေးစနစ် မဖြစ်မနေ ထည့်သွင်းတပ်ဆင်ရမယ်လို့ စည်ပင်သာယာရေးကော်မတီ အဆောက်အအုံအင်ဂျင်နီယာဌာနက ညွှန်ကြားချက် ထုတ်ပြန်ပါတယ်။ ဆောက်လုပ်ရေးပစ္စည်းတွေ ဈေးကြီး၊ လုပ်ငန်းအများစု ရပ်နားထားရချိန်နဲ့ ဆိုလာနဲ့ ဆက်စပ်ပစ္စည်းတွေ ဈေးအဆမတန် တက်နေချိန်မှာ ဈေးကွက်ကို မထိန်းညှိနိုင်ဘဲ ဆိုလာစနစ် မဖြစ်မနေ ထည့်သွင်းတပ်ဆင်ခိုင်းတာက ဘယ်လောက်ထိ အခက်အခဲ ဖြစ်စေနိုင်ပါသလဲ။ မဖြစ်မနေ တပ်ဆင်ရမယ်ဆိုရင် ဘာတွေလုပ်ဖို့ လိုအပ်ပါသလဲ။ ဆိုလာနည်းပညာ ကျွမ်းကျင်သူတွေနဲ့ ဆောက်လုပ်ရေး လုပ်ငန်းရှင်တွေက ဒီအပေါ်မှာ ဘယ်လို ရှုမြင်ကြပါသလဲ။ အပြည့်အစုံကို မသူဇာလင်းက ပြောပြမှာပါ။",
    },
    "🎙️ Alex": {
        "audio_path": "/tmp/alex_voice.wav",
        "prompt_text": "Make heads or tails of this. You say this when you can't understand something because it's confusing. You say, I've read this three times and I still can't make heads or tails of it. I'm as much in the dark as you are. You say this when you know just as little about a situation as the other person. You say, why was the meeting cancelled? I'm as much in the dark as you are. I think it's worth a shot. You say this when you think something is worth trying even if it might not work. You say, we might get rejected, but I think it's worth a shot. If I play my cards right, you say this when you believe you can get what you want by making the right choices. You say, if I play my cards right I could get that promotion. We hit it off right away. You say this when two people like each other and get along from the beginning. You say, I was nervous about meeting her, but we hit it off right away. Take a rain check on that. You say this when you want to postpone an invitation or activity until another time. You say, I'd love to, but can I take a rain check on that? I'm exhausted.",
    },
}



In [ ]:
# ── Prompt cache: reference voice ကို တစ်ခါတည်း encode, အပိုင်းတိုင်း ပြန်သုံး ──
# (Voice-Studio backend pattern — thelapyae/Voice-Studio)
_PROMPT_CACHE = {}

def get_prompt_cache(audio_path, prompt_text):
    key = (str(audio_path), str(prompt_text or ""))
    pc = _PROMPT_CACHE.get(key)
    if pc is None:
        pc = model.tts_model.build_prompt_cache(
            prompt_text=prompt_text,
            prompt_wav_path=audio_path,
            reference_wav_path=audio_path,
        )
        _PROMPT_CACHE[key] = pc
    return pc

def _to_numpy(wav):
    if hasattr(wav, "detach"):
        wav = wav.detach().cpu()
        if hasattr(wav, "squeeze"):
            wav = wav.squeeze(0)
        wav = wav.numpy()
    return np.asarray(wav, dtype=np.float32)

def run_tts(text_input, audio_path, prompt_text):
    gc.collect(); torch.cuda.empty_cache()
    smart = (text_input.replace('။', '။\n').replace('.', '.\n')
                       .replace('?', '?\n').replace('!', '!\n'))
    chunks = [t.strip() for t in smart.split('\n') if t.strip()]
    if not chunks:
        return None
    # Prompt cache ရမရ စမ်း (VoxCPM version အလိုက် API ကွဲနိုင် → မရရင် ပုံမှန်လမ်း)
    try:
        _pc = get_prompt_cache(audio_path, prompt_text)
        _use_cache = hasattr(model.tts_model, "generate_with_prompt_cache")
        if _use_cache:
            print("⚡ prompt cache သုံးမယ် (reference တစ်ခါတည်း encode)")
    except Exception as e:
        print(f"⚠️ prompt cache မရ ({e}) — ပုံမှန် generate သုံးမယ်")
        _use_cache = False
    silence = np.zeros(int(SR * 0.15), dtype=np.float32)
    wavs = []
    for i, ch in enumerate(chunks):
        if len(ch) < 2:
            continue
        # seed ချုပ် → run တိုင်း အသံတူ (model.generate က seed kwarg မရ)
        torch.manual_seed(42); torch.cuda.manual_seed_all(42); np.random.seed(42)
        with torch.inference_mode():
            if _use_cache:
                wt, _, _ = model.tts_model.generate_with_prompt_cache(
                    target_text=ch + " ",
                    prompt_cache=_pc,
                    inference_timesteps=15,
                    cfg_value=2.1,
                    max_len=2048,
                )
                w = _to_numpy(wt)
            else:
                w = model.generate(text=ch + " ", prompt_wav_path=audio_path,
                                   prompt_text=prompt_text,
                                   cfg_value=2.1, inference_timesteps=15)
        wavs.append(w)
        if i < len(chunks) - 1:
            wavs.append(silence)
        torch.cuda.empty_cache(); gc.collect()
    return np.concatenate(wavs)


In [ ]:
import re as _re

def _fname_from_text(text, ts):
    """Output filename: first ~6 words of the input text + timestamp."""
    words = _re.findall(r"[A-Za-z0-9']+", str(text or ""))[:6]
    base = "_".join(words) if words else "voice"
    return f"{base}_{ts}.wav"

def generate_preset(text_input, voice_choice):
    os.makedirs("/tmp/voxcpm_outputs", exist_ok=True)
    sel = VOICE_DATABASE.get(voice_choice, list(VOICE_DATABASE.values())[0])
    wav = run_tts(text_input, sel["audio_path"], sel["prompt_text"])
    if wav is None:
        return None
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = f"/tmp/voxcpm_outputs/{_fname_from_text(text_input, ts)}"
    sf.write(out, wav, SR)
    return out

def generate_clone(text_input, uploaded_audio, reference_text):
    if uploaded_audio is None:
        return None, "⚠️ အသံဖိုင် တင်ပေးပါ။"
    if not (reference_text or "").strip():
        return None, "⚠️ Reference Text ထည့်ပေးပါ။"
    wav = run_tts(text_input, uploaded_audio, reference_text.strip())
    if wav is None:
        return None, "⚠️ စာသား မပါပါ။"
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = f"/tmp/voxcpm_outputs/{_fname_from_text(text_input, ts)}"
    sf.write(out, wav, SR)
    return out, "✅ အသံထုတ်ပြီးပါပြီ!"



In [ ]:
with gr.Blocks(title="🎙️ Channel 99 AI Studio") as demo:
    gr.Markdown("# 🎙️ Channel 99 AI Studio")
    with gr.Tabs():
        with gr.Tab("🎭 Preset Voice"):
            with gr.Row():
                with gr.Column():
                    t1 = gr.Textbox(lines=10, label="📝 စာသားထည့်ပါ")
                    v1 = gr.Radio(choices=list(VOICE_DATABASE.keys()),
                                  value=list(VOICE_DATABASE.keys())[0],
                                  label="🗣️ အသံရွေးပါ")
                    b1 = gr.Button("🎙️ အသံထုတ်မယ်", variant="primary")
                with gr.Column():
                    o1 = gr.Audio(label="🎧 ထွက်လာမယ့် အသံ")
            b1.click(fn=generate_preset, inputs=[t1, v1], outputs=o1)
        with gr.Tab("🎤 Clone Voice"):
            with gr.Row():
                with gr.Column():
                    t2 = gr.Textbox(lines=10, label="📝 စာသားထည့်ပါ")
                    a2 = gr.Audio(label="🎵 Reference အသံဖိုင် (WAV)", type="filepath")
                    r2 = gr.Textbox(lines=4, label="📄 Reference Text (အတိအကျ)")
                    b2 = gr.Button("🎤 Clone ပြီး ထုတ်မယ်", variant="primary")
                with gr.Column():
                    o2 = gr.Audio(label="🎧 ထွက်လာမယ့် အသံ")
                    s2 = gr.Textbox(label="Status", interactive=False)
            b2.click(fn=generate_clone, inputs=[t2, a2, r2], outputs=[o2, s2])

import threading, time

print("\n🚀 Channel 99 AI Studio — 2 tabs: Preset Voice | Clone Voice")
print("📌 Colab tab ကို ဖွင့်ထားပါ (ပိတ်လိုက်ရင် server ရပ်မယ်)\n")


def _launch():
    demo.launch(share=True, debug=False, prevent_thread_lock=True)

t = threading.Thread(target=_launch, daemon=True)
t.start()

share_url = None
for _ in range(60):  # max ~5 min စောင့်
    time.sleep(5)
    if getattr(demo, "share_url", None):
        share_url = demo.share_url.strip()
        break

if share_url:
    print("\n" + "="*60 + "\n📋 COPY THIS URL → recap tool:\n" + share_url + "\n" + "="*60)
else:
    print("⚠️ share URL မရသေး — output အပေါ်က 'Running on public URL' ကို ကြည့်")

t.join()  # server ဆက်ရှင်နေအောင်


In [ ]:
# ── Share URL ကို ဒီမှာ ကြည့်ပါ ──
# အပေါ် cell ရဲ့ output မှာ "Running on public URL: https://xxxxx.gradio.live" ပါတယ်
# အဲ့ URL ကို copy → recap tool Config → VoxCPM URL ထဲ ကပ်
